# Notebook 06 — Rating Distribution

**Project:** Starbucks Customer Voice Intelligence: A U.S. Coffee Chain Market Study  
**Purpose:** Break down Starbucks' star rating distribution, compare it to the market, and check whether the gap has grown over time.

Starbucks averages 2.90 stars. The coffee market averages 3.97. That is a 1-point gap on a 5-point scale, and it has been there for years. This notebook looks at where it comes from.

## 0. Environment setup

In [1]:
import sys
import pandas as pd
import plotly.graph_objects as go
from pathlib import Path

PROJECT_ROOT = Path(".").resolve().parent
sys.path.insert(0, str(PROJECT_ROOT))

INTERIM_DIR = PROJECT_ROOT / "data" / "processed"
FIGURES_DIR = PROJECT_ROOT / "outputs" / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print("Environment ready.")

Environment ready.


## 1. Load data

In [2]:
df     = pd.read_parquet(INTERIM_DIR / "analysis_ready.parquet")
sbux   = df[df["brand_category"] == "Starbucks"]
market = df[df["brand_category"] != "Starbucks"]

print(f"Starbucks reviews : {len(sbux):,}")
print(f"Market reviews    : {len(market):,}")
print(f"\nStarbucks avg rating : {sbux['review_stars'].mean():.2f}")
print(f"Market avg rating    : {market['review_stars'].mean():.2f}")
print(f"\nStarbucks star rating counts:")
print(sbux["review_stars"].value_counts().sort_index().to_string())

Starbucks reviews : 11,675
Market reviews    : 370,324

Starbucks avg rating : 2.90
Market avg rating    : 3.97

Starbucks star rating counts:
review_stars
1.0    3937
2.0    1620
3.0    1171
4.0    1562
5.0    3385


## 2. Star rating distribution

Star rating counts across all Starbucks reviews. The shape tells you whether the problem is a flood of 1-stars, a weak middle, or something else.

In [3]:
star_counts = sbux["review_stars"].value_counts().sort_index()
star_pcts   = (star_counts / len(sbux) * 100).round(1)

fig = go.Figure(go.Pie(
    labels=[f"{int(s)}\u2605" for s in star_counts.index],
    values=star_counts.values,
    marker=dict(colors=["#d62728", "#ff7f0e", "#bcbd22", "#2ca02c", "#1f77b4"]),
    textinfo="label+percent",
    hole=0.3,
))
fig.update_layout(
    title=dict(text="Starbucks — Star Rating Distribution", font=dict(size=16)),
    paper_bgcolor="white",
    width=600, height=480,
    margin=dict(t=60, b=20, l=20, r=20),
)
fig.write_html(str(FIGURES_DIR / "06_star_distribution.html"))
fig.show()

## 3. Satisfaction tier breakdown

Reviews grouped into Critical (1–2 stars), Neutral (3 stars), and Positive (4–5 stars). A simpler read on the five-point scale, and more directly tied to retention risk.

In [4]:
tier_order  = ["Critical", "Neutral", "Positive"]
tier_colors = {"Critical": "#d62728", "Neutral": "#ff7f0e", "Positive": "#2ca02c"}
tier_counts = sbux["star_tier"].value_counts()

print("Starbucks star tier breakdown:")
for t in tier_order:
    n = tier_counts.get(t, 0)
    print(f"  {t:<10} {n:>5,}  ({n/len(sbux)*100:.1f}%)")

fig = go.Figure(go.Pie(
    labels=tier_order,
    values=[tier_counts.get(t, 0) for t in tier_order],
    marker=dict(colors=[tier_colors[t] for t in tier_order]),
    textinfo="label+percent",
    hole=0.3,
))
fig.update_layout(
    title=dict(text="Starbucks — Satisfaction Tier Breakdown", font=dict(size=16)),
    paper_bgcolor="white",
    width=600, height=480,
    margin=dict(t=60, b=20, l=20, r=20),
)
fig.write_html(str(FIGURES_DIR / "06_star_tier.html"))
fig.show()

Starbucks star tier breakdown:
  Critical   5,557  (47.6%)
  Neutral    1,171  (10.0%)
  Positive   4,947  (42.4%)


## 4. Rating trend vs. market

Starbucks' average rating by year, with the market average as a grey reference line.

In [5]:
sbux_avg   = sbux.groupby("year")["review_stars"].mean().round(2)
market_avg = market.groupby("year")["review_stars"].mean().round(2)

print("Year  Starbucks  Market  Gap")
for yr in sbux_avg.index:
    gap = market_avg[yr] - sbux_avg[yr]
    print(f"{int(yr)}  {sbux_avg[yr]:.2f}       {market_avg[yr]:.2f}    {gap:.2f}")

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=sbux_avg.index.astype(str), y=sbux_avg.values,
    mode="lines+markers", name="Starbucks",
    line=dict(color="#00704A", width=2.5),
    marker=dict(size=8),
))
fig.add_trace(go.Scatter(
    x=market_avg.index.astype(str), y=market_avg.values,
    mode="lines+markers", name="Market avg",
    line=dict(color="#aaaaaa", width=2, dash="dash"),
    marker=dict(size=7),
))
fig.update_layout(
    title=dict(text="Average Star Rating by Year — Starbucks vs. Market", font=dict(size=16)),
    xaxis_title="Year", yaxis_title="Avg Star Rating",
    plot_bgcolor="white", paper_bgcolor="white",
    xaxis=dict(showgrid=False),
    yaxis=dict(showgrid=True, gridcolor="#eeeeee", range=[1, 5]),
    legend=dict(x=0.75, y=0.15),
    width=820, height=460,
    margin=dict(t=60, b=50, l=60, r=40),
)
fig.write_html(str(FIGURES_DIR / "06_rating_trend.html"))
fig.show()

Year  Starbucks  Market  Gap
2017  2.99       3.94    0.95
2018  2.93       3.96    1.03
2019  2.89       3.96    1.07
2020  2.88       4.04    1.16
2021  2.76       3.97    1.21


In [6]:

# Sentiment score trend over time — Starbucks vs market
sbux_sent   = sbux.groupby("year")["sentiment_score"].mean().round(3)
market_sent = market.groupby("year")["sentiment_score"].mean().round(3)

fig_s = go.Figure()
fig_s.add_trace(go.Scatter(
    x=sbux_sent.index.astype(str), y=sbux_sent.values,
    mode="lines+markers", name="Starbucks",
    line=dict(color="#00704A", width=2.5), marker=dict(size=8),
))
fig_s.add_trace(go.Scatter(
    x=market_sent.index.astype(str), y=market_sent.values,
    mode="lines+markers", name="Market",
    line=dict(color="#aaaaaa", width=2, dash="dot"), marker=dict(size=6),
))
fig_s.update_layout(
    title=dict(text="Starbucks vs Market — Avg VADER Sentiment Score (2017–2021)", font=dict(size=16)),
    xaxis_title="Year",
    yaxis_title="Avg Sentiment Score",
    plot_bgcolor="white", paper_bgcolor="white",
    xaxis=dict(showgrid=False),
    yaxis=dict(showgrid=True, gridcolor="#eeeeee"),
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
    width=720, height=420,
    margin=dict(t=70, b=50, l=60, r=40),
)
fig_s.write_html(str(FIGURES_DIR / "06_sentiment_trend.html"))
fig_s.show()

print("\nSentiment score by year:")
print(pd.DataFrame({"Starbucks": sbux_sent, "Market": market_sent}).to_string())



Sentiment score by year:
      Starbucks  Market
year                   
2017      0.301   0.701
2018      0.308   0.696
2019      0.277   0.691
2020      0.264   0.696
2021      0.207   0.671


## Key findings

- The distribution is U-shaped: 1-star is the largest single group (33.7%), 5-star is second (29.0%). The three middle ratings together account for 37.3%.
- 47.6% of Starbucks reviews are Critical vs. 42.4% Positive. More dissatisfied customers on record than satisfied ones.
- The market held steady at 3.94–4.04 across all five years. Starbucks fell from 2.99 to 2.76, and the gap with the market widened from 0.95 to 1.21.
- The decline started in 2017, well before COVID. It has not reversed in any year since, which points to something structural.
- VADER sentiment and star ratings diverge in 24.0% of Starbucks reviews. Most of the mismatch runs in one direction: 22.6% of reviews carry a low star rating (1–3) but positive sentiment language. These customers are disappointed but not angry — a different recovery challenge than those who are both low-rated and negative in tone.

---

**Next: Notebook 07 — Voice of Customer: Loyalty Drivers**

## 5. Star rating vs. sentiment score

Star ratings and sentiment scores measure different things. The table below shows how VADER sentiment tracks against each star level — and where the two signals diverge.

In [7]:
tbl = sbux.groupby("review_stars").agg(
    count=("review_id", "count"),
    avg_sentiment=("sentiment_score", "mean"),
).reset_index()
tbl["pct"]           = (tbl["count"] / len(sbux) * 100).round(1)
tbl["avg_sentiment"] = tbl["avg_sentiment"].round(3)
tbl["review_stars"]  = tbl["review_stars"].astype(int)
tbl.columns = ["Stars", "Count", "Avg Sentiment Score", "% of Reviews"]
tbl = tbl[["Stars", "Count", "% of Reviews", "Avg Sentiment Score"]]

print("Star rating vs. avg VADER sentiment score (Starbucks):")
print(tbl.to_string(index=False))

high_star_neg = ((sbux["review_stars"] >= 4) & (sbux["sentiment_label"] == "Negative")).sum()
low_star_pos  = ((sbux["review_stars"] <= 3) & (sbux["sentiment_label"] == "Positive")).sum()

print(f"\nStar-sentiment mismatch:")
print(f"  Low star (1-3)  + Positive sentiment : {low_star_pos:,}  ({low_star_pos/len(sbux)*100:.1f}%)")
print(f"  High star (4-5) + Negative sentiment : {high_star_neg:,}  ({high_star_neg/len(sbux)*100:.1f}%)")

Star rating vs. avg VADER sentiment score (Starbucks):
 Stars  Count  % of Reviews  Avg Sentiment Score
     1   3937          33.7               -0.290
     2   1620          13.9               -0.019
     3   1171          10.0                0.333
     4   1562          13.4                0.726
     5   3385          29.0                0.851

Star-sentiment mismatch:
  Low star (1-3)  + Positive sentiment : 2,633  (22.6%)
  High star (4-5) + Negative sentiment : 163  (1.4%)
